Khushi Khatri BEB222 Experiment 7
Aim: Perform chunking for the given dataset.

Theory

Chunking (also called shallow parsing) is the process of grouping
POS-tagged words into larger, meaningful phrases — such as noun phrases
(NP), verb phrases (VP), and prepositional phrases (PP) — without building
a full, deeply-nested parse tree. It sits one level above POS tagging
(Experiment 6): where POS tagging labels each individual word, chunking
groups sequences of those tags into flat, non-overlapping chunks.

Chunking is useful because most NLP applications don't need a complete
grammatical parse tree, just the key phrases. For sentiment analysis
specifically, noun-phrase chunks are what let us find the *aspects*
being talked about (e.g. "the host", "the location", "a great view"),
while verb-phrase chunks capture the *actions/opinions* attached to them
(e.g. "was extremely helpful"). This is the basis of aspect-based
sentiment analysis, where a review isn't just scored positive/negative
overall but broken down by what specifically was liked or disliked.


In [1]:
import pandas as pd
from collections import Counter
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize

print('Setup complete.')

Setup complete.


[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [2]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


chunk_grammar = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}
  PP: {<IN><NP>}
  VP: {<VB.*><RB.*>?}
"""

chunk_parser = nltk.RegexpParser(chunk_grammar)
print(chunk_parser)

In [4]:
def chunk_text(text):
    """Tokenize -> POS tag -> chunk. Returns an nltk Tree."""
    if not isinstance(text, str) or text.strip() == '':
        return None
    tokens = word_tokenize(text)
    tagged = nltk.pos_tag(tokens)
    return chunk_parser.parse(tagged)


# Demo on a sample review
sample_text = df['review_text'].iloc[0]
print('Original Text:\n', sample_text, '\n')

sample_tree = chunk_text(sample_text)
print(sample_tree)

Original Text:
 Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed. 

(S
  (VP Amazing/VBG)
  (NP stay/NN)
  !/.
  (NP The/DT place/NN)
  (VP felt/VBD very/RB)
  cozy/JJ
  for/IN
  4/CD
  (NP guests/NNS)
  ./.
  (NP Check-in/NNP)
  (VP was/VBD)
  smooth/JJ
  and/CC
  (NP the/DT amenities/NNS)
  (VP were/VBD exactly/RB)
  what/WP
  we/PRP
  (VP needed/VBD)
  ./.)


In [5]:
sample_tree.pretty_print()

                                                                               S                                                                                                                                        
  _____________________________________________________________________________|__________________________________________________________________________________________________________________________________       
 |     |      |     |    |      |       |       |      |     |       VP        NP           NP                    VP             NP          NP         VP           NP                         VP                VP    
 |     |      |     |    |      |       |       |      |     |       |         |       _____|_____          ______|_____         |           |          |       _____|________            ______|______           |      
!/. cozy/JJ for/IN 4/CD ./. smooth/JJ and/CC what/WP we/PRP ./. Amazing/VBG stay/NN The/DT     place/NN felt/VBD     very/RB guest

In [6]:
def extract_chunks(tree, label):
    """Return a list of chunk phrases (as strings) for the given label, e.g. 'NP'."""
    if tree is None:
        return []
    return [
        ' '.join(word for word, tag in subtree.leaves())
        for subtree in tree.subtrees()
        if subtree.label() == label
    ]


noun_phrases = extract_chunks(sample_tree, 'NP')
verb_phrases = extract_chunks(sample_tree, 'VP')
prep_phrases = extract_chunks(sample_tree, 'PP')

print('Noun Phrases:', noun_phrases)
print('Verb Phrases:', verb_phrases)
print('Prepositional Phrases:', prep_phrases)

Noun Phrases: ['stay', 'The place', 'guests', 'Check-in', 'the amenities']
Verb Phrases: ['Amazing', 'felt very', 'was', 'were exactly', 'needed']
Prepositional Phrases: []


In [7]:
def chunk_row(text):
    tree = chunk_text(text)
    return pd.Series({
        'noun_phrases': extract_chunks(tree, 'NP'),
        'verb_phrases': extract_chunks(tree, 'VP'),
    })


sample_df = df.head(1000).copy()
chunk_results = sample_df['review_text'].apply(chunk_row)
df_chunks = pd.concat([sample_df, chunk_results], axis=1)

df_chunks[['review_id', 'review_text', 'noun_phrases', 'verb_phrases']].head(10)

,review_id,review_text,noun_phrases,verb_phrases
0,369314882,Amazing stay! The place felt very cozy for 4 g...,"[stay, The place, guests, Check-in, the amenit...","[Amazing, felt very, was, were exactly, needed]"
1,490116563,It was okay for the price. Location in XIII Au...,"[the price, Location, XIII Aurelia]","[was, okay, was]"
2,582235668,Loved every minute of it. Our superhost was su...,"[every minute, superhost, super responsive, Gr...","[Loved, was, communicate, needed]"
3,68054683,Decent stay overall. Some things could be impr...,"[Decent stay, Some things, cleanliness, the en...","[be, improved]"
4,248483824,Reasonable for a short trip. Location in Long ...,"[a short trip, Location, Long Island City]",[was]
5,155617131,Decent stay overall. It served its purpose for...,"[Decent stay, purpose, stay, Paris]",[served]
6,710244614,"Nothing special, but fine. Our superhost was p...","[superhost, a couple, times, Some things, clea...","[Nothing, was, respond, be, improved]"
7,299174484,We had a rough experience. The location in Enc...,"[a rough experience, The location, Enclos-St-L...","[had, was, expected, had, were never, resolved..."
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[Perfect, trip, Check-in, the amenities, The p...","[was, were exactly, needed, felt very, needed]"
9,469473761,Would not recommend. The private room in house...,"[The private room, house, the photos, several ...","[recommend, was not, suggested, had, were neve..."


In [8]:
all_noun_phrases = Counter([np for row in df_chunks['noun_phrases'] for np in row])
all_verb_phrases = Counter([vp for row in df_chunks['verb_phrases'] for vp in row])

print('Top 15 Noun Phrases across dataset:')
for phrase, count in all_noun_phrases.most_common(15):
    print(phrase, '->', count)

print('\nTop 15 Verb Phrases across dataset:')
for phrase, count in all_verb_phrases.most_common(15):
    print(phrase, '->', count)

Top 15 Noun Phrases across dataset:
stay -> 380
Check-in -> 290
The host -> 230
The entire apartment -> 229
superhost -> 189
issues -> 169
the photos -> 167
several issues -> 161
The location -> 157
the price -> 156
arrival -> 151
super responsive -> 146
Great location -> 144
everything -> 144
the amenities -> 139

Top 15 Verb Phrases across dataset:
was -> 1299
had -> 438
needed -> 283
expected -> 247
reach -> 169
was not -> 167
suggested -> 167
were never -> 161
resolved -> 161
confusing -> 151
delayed -> 151
communicate -> 146
were exactly -> 139
felt very -> 134
served -> 122


In [9]:
df_chunks.to_csv('Chunked_Airbnb_Reviews.csv', index=False)
print('Saved to Chunked_Airbnb_Reviews.csv')
print(df_chunks.shape)

Saved to Chunked_Airbnb_Reviews.csv
(1000, 44)
